# Variant to mutator background
Find in which mutator each variant occured.

D1-D4 are complicated attempts at tracking the subclades, but this is not reliable. Instead, here is a simple version:
* If the mutator allele was detectable when the variant was first detected, considered it occured in the mutator background
* If the variant occured when the reversion was near complete, considered it occured in the reverted background

However, using these heuristics, it appears that many variants attributed to the *reverted background* in fact appeared in the *mutator background*.
We can tell for two reasons:
* The apparent mutation rate is higher immediately after reversion, and only goes back down to the original rate after a delay
* The mutation spectrum immediately after reversion is also more similar to that of mutators

What's happening is that, at the time of reversion, many cells in the population already had some recent, very low-frequency mutations.
These were not detectable at the time.
However, once evolution allowed for some of these cells to grow in frequency, the recent mutations they carried also grew and became visible, giving the impression that they occured after the mutator's reversion.

Therefore, a variant will be considered to have occured in a mutator if *any of the previous timepoints* (2,500 gen) contained a mutator.

In [1]:
source('../../../R/preamble.R')

freqplot = function(df, spacing = 0.00) {
    ggplot(df) + aes(gen, freq + spacing*dense_rank(pos), col = pos, group = tid) + 
    geom_line() + geom_point(size = 0.3) +
    #scale_x_continuous(breaks = 0:16) +
    facet_wrap(~pop, ncol = 4) +
    labs(x = 'Generation', y = 'Frequency')
}

d = read.tsv('../../../datasets/genomics/04_gene_attribution/results/d_labeled.tsv')

mut_vids = c('mutS_1', 'mutL_1', 'ygbA/mutS_1')
rev_vids = c('mutS_2')

In [2]:
mut_thr = 0
rev_thr = 1

mutator_labels = c('WT' = 'None', 'Reverted' = 'Reverted', 'mutL_1' = 'mutL(Δ6)', 'mutS_1' = 'mutS(G696E)', 'ygbA/mutS_1' = 'pMutS')

d_with_status =
d %>%

    group_by(pop, gen) %>% 
    mutate(
        mut_freq = ifelse(any(vid %in% mut_vids), freq[vid %in% mut_vids], 0),
        rev_freq = ifelse(any(vid %in% rev_vids), freq[vid %in% rev_vids], 0)
        ) %>% 

    group_by(tid) %>%
    mutate(
        mut_freq_at_apparition = max(mut_freq[gen >= apparition - 1 & gen <= apparition], na.rm = T),
        rev_freq_at_apparition = min(rev_freq[gen >= apparition - 1 & gen <= apparition], na.rm = T),
        mutator_status = case_when(
            rev_freq_at_apparition >= rev_thr ~ 'Reverted',
            mut_freq_at_apparition > mut_thr ~ 'Mutator',
            T ~ 'WT'
            )
        ) %>%

    group_by(pop) %>%
    mutate(
        mutator_vid = ifelse(mutator_status == 'Mutator', vid[vid %in% mut_vids], mutator_status),
        mutator_bg = mutator_labels[mutator_vid]
    ) %>% 

    ungroup()

# Export
Reminder: tid is a variant trajectory id (there is one for each variant that appears).

In [3]:
tid_to_mutator_bg = 
    d_with_status %>%
    select(pop, tid, mutator_status, mutator_vid, mutator_bg) %>% 
    unique()

tid_to_mutator_bg %>% write.tsv('../../../datasets/genomics/06_mutators/results/tid_to_mutator_bg.tsv')

# Dominant mutator at each timepoint

In [4]:
pop_gen_to_dominant_mutator_bg = d_with_status %>%
    filter(gen == apparition) %>% 
    select(pop, gen, dominant_mutator_vid = mutator_vid, dominant_mutator_status = mutator_status, dominant_mutator_bg = mutator_bg) %>%
    unique() %>%
    arrange(pop, gen)

pop_gen_to_dominant_mutator_bg %>% write.tsv('../../../datasets/genomics/06_mutators/results/pop_gen_to_dominant_mutator_bg.tsv')

# Standardize nucleotide substitutions
E.g. A -> C is the same as T -> G in terms of mutation spectrum.

In [5]:
# Standardize nucleotide substitutions
revcom = c('A' = 'T', 'T' = 'A', 'C' = 'G', 'G' = 'C')

ref_alt_to_nucleotide_transition =
d %>%
    filter(type == 'SNP') %>% select(ref, alt) %>% unique() %>% 

    # Flip so the original is always A or C
    mutate(
        ref_flip = case_when(ref == 'T' ~ 'A', ref == 'G' ~ 'C', T ~ ref),
        alt_flip = ifelse(ref != ref_flip, revcom[alt], alt),
        ref_pair = paste(ref_flip, revcom[ref_flip], sep = '-'),
        alt_pair = paste(alt_flip, revcom[alt_flip], sep = '-'),
        transition = paste(ref_pair, alt_pair, sep = '→')
    ) %>%

    select(ref, alt, transition)

write.tsv(ref_alt_to_nucleotide_transition, '../../../datasets/genomics/06_mutators/results/ref_alt_to_nucleotide_transition.tsv')